# 朝日新聞の記事検索結果を取得する

朝日新聞デジタルで「能登半島地震」を検索し、検索欄の合計件数までタイトル・本文・公開日時・URLを取得します。

- ログインせずに検索し、公開範囲で閲覧できる本文のみ取得します。
- 短時間に大量アクセスしないよう待機時間を設けています。
- 実行前に利用規約・著作権・robots.txtを確認し、取得データは許可された範囲で利用してください。


In [1]:
# 初回だけ実行してください
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "requests", "beautifulsoup4", "pandas"
])



[notice] A new release of pip is available: 24.3.1 -> 26.2
[notice] To update, run: pip install --upgrade pip


0

In [2]:
import json
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.asahi.com"
SEARCH_API_URL = "https://sitesearch.asahi.com/sitesearch-api/"
KEYWORD = "能登半島地震"

REQUEST_INTERVAL = 1.5
TIMEOUT = 30

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0 Safari/537.36"
    ),
    "Accept-Language": "ja,en-US;q=0.8,en;q=0.6",
    "Accept-Encoding": "gzip, deflate",
})


In [3]:
def get_soup(url):
    response = session.get(url, timeout=TIMEOUT)
    response.raise_for_status()
    return BeautifulSoup(response.content, "html.parser")


def search_articles(keyword):
    """検索APIを次の開始位置へ進め、表示された合計件数まで取得する。"""
    found = []
    seen = set()
    start = 0
    total_count = None

    while True:
        response = session.get(
            SEARCH_API_URL,
            params={"Keywords": keyword, "start": start, "sort": 1},
            timeout=TIMEOUT,
        )
        response.raise_for_status()
        goo = response.json().get("goo") or {}
        docs = goo.get("docs") or []

        if total_count is None:
            hit_num = goo.get("hit_num") or {}
            total_count = int(hit_num.get("num") or 0)
            print(f"検索結果の合計件数: {total_count:,} 件")

        new_count = 0
        for doc in docs:
            url = str(doc.get("URL") or "").strip()
            if not url or url in seen:
                continue
            seen.add(url)
            found.append({
                "title": str(doc.get("TITLE") or "").strip(),
                "published_at": doc.get("ReleaseDate"),
                "url": url,
                "search_excerpt": str(doc.get("BODY") or "").strip(),
            })
            new_count += 1

        print(f"検索一覧を取得中: {len(found):,} / {total_count:,} 件")
        if len(found) >= total_count:
            break
        if not docs or new_count == 0:
            print(
                f"警告: 合計 {total_count:,} 件のうち {len(found):,} 件で"
                "次の検索結果がなくなりました。"
            )
            break

        next_info = ((goo.get("paging") or {}).get("navi") or {}).get("next")
        if next_info and "start" in next_info:
            start = int(next_info["start"])
        else:
            start += len(docs)
        time.sleep(REQUEST_INTERVAL)

    return found


def news_article_json_ld(soup):
    for script in soup.select('script[type="application/ld+json"]'):
        try:
            data = json.loads(script.string or script.get_text())
        except (json.JSONDecodeError, TypeError):
            continue
        candidates = data if isinstance(data, list) else [data]
        for candidate in candidates:
            if isinstance(candidate, dict) and candidate.get("@type") in {
                "NewsArticle", "Article"
            }:
                return candidate
    return {}


def parse_article_page(soup, search_row):
    metadata = news_article_json_ld(soup)

    # 現行記事ページの本文領域。class名変更時に備えてmain内もフォールバックする。
    body_scope = soup.select_one("div.nfyQp")
    paragraph_nodes = body_scope.select("p") if body_scope else []
    paragraphs = []
    for node in paragraph_nodes:
        text = node.get_text(" ", strip=True)
        if not text:
            continue
        if "無断転載を禁じます" in text or "著作権法" in text:
            continue
        paragraphs.append(text)

    body = "\n\n".join(dict.fromkeys(paragraphs))
    body_is_excerpt = False
    page_text = soup.get_text(" ", strip=True)
    if "この記事は有料会員記事です" in page_text or "残り：" in page_text:
        body_is_excerpt = True

    if not body:
        body = str(metadata.get("description") or search_row["search_excerpt"]).strip()
        body_is_excerpt = bool(body)

    return {
        "title": metadata.get("headline") or search_row["title"],
        "body": body,
        "published_at": (
            metadata.get("datePublished") or search_row["published_at"]
        ),
        "url": search_row["url"],
        "body_is_excerpt": body_is_excerpt,
    }


def collect_articles(keyword):
    search_rows = search_articles(keyword)
    print(f"検索結果から {len(search_rows):,} 件のURLを取得しました。")

    articles = []
    for index, row in enumerate(search_rows, start=1):
        print(f"[{index:,}/{len(search_rows):,}] {row['title']}")
        try:
            soup = get_soup(row["url"])
            articles.append(parse_article_page(soup, row))
        except (requests.RequestException, ValueError) as exc:
            articles.append({
                "title": row["title"],
                "body": row["search_excerpt"],
                "published_at": row["published_at"],
                "url": row["url"],
                "body_is_excerpt": True,
                "error": str(exc),
            })
        if index < len(search_rows):
            time.sleep(REQUEST_INTERVAL)

    return articles


In [5]:
articles = collect_articles(KEYWORD)

df = pd.DataFrame(articles)
if not df.empty:
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    df = df.sort_values("published_at", ascending=False, na_position="last")
df = df.reset_index(drop=True)
df.insert(0, "ID", range(1, len(df) + 1))

df


検索結果の合計件数: 7,726 件
検索一覧を取得中: 20 / 7,726 件
検索一覧を取得中: 40 / 7,726 件
検索一覧を取得中: 60 / 7,726 件
検索一覧を取得中: 80 / 7,726 件
検索一覧を取得中: 100 / 7,726 件
検索一覧を取得中: 120 / 7,726 件
検索一覧を取得中: 140 / 7,726 件
検索一覧を取得中: 160 / 7,726 件
検索一覧を取得中: 180 / 7,726 件
検索一覧を取得中: 199 / 7,726 件
検索一覧を取得中: 216 / 7,726 件
検索一覧を取得中: 236 / 7,726 件
検索一覧を取得中: 252 / 7,726 件
検索一覧を取得中: 272 / 7,726 件
検索一覧を取得中: 291 / 7,726 件
検索一覧を取得中: 307 / 7,726 件
検索一覧を取得中: 327 / 7,726 件
検索一覧を取得中: 340 / 7,726 件
検索一覧を取得中: 357 / 7,726 件
検索一覧を取得中: 373 / 7,726 件
検索一覧を取得中: 389 / 7,726 件
検索一覧を取得中: 402 / 7,726 件
検索一覧を取得中: 416 / 7,726 件
検索一覧を取得中: 430 / 7,726 件
検索一覧を取得中: 443 / 7,726 件
検索一覧を取得中: 459 / 7,726 件
検索一覧を取得中: 470 / 7,726 件
検索一覧を取得中: 481 / 7,726 件
検索一覧を取得中: 495 / 7,726 件
検索一覧を取得中: 509 / 7,726 件
検索一覧を取得中: 526 / 7,726 件
検索一覧を取得中: 546 / 7,726 件
検索一覧を取得中: 564 / 7,726 件
検索一覧を取得中: 580 / 7,726 件
検索一覧を取得中: 596 / 7,726 件
検索一覧を取得中: 603 / 7,726 件
検索一覧を取得中: 606 / 7,726 件
検索一覧を取得中: 626 / 7,726 件
検索一覧を取得中: 642 / 7,726 件
検索一覧を取得中: 662 / 7,726 件
検索一覧を取得中: 682 / 7,726 件
検

,ID,title,body,published_at,url,body_is_excerpt,error
0,1,続く猛暑で命を守るためには 10年前の熊本地震を知る救命医に聞く,熊本地震 の被災地では連日、厳しい暑さが続いています。発災から1週間。猛暑のなか、命を守るた...,2026-08-04 13:00:00+09:00,https://www.asahi.com/articles/ASV831GF3V83UPQ...,False,NaN
1,2,（２０２６年熊本地震）地震、予備費２００億円超支出へ 首相が熊本視察、「激甚災害」指定も,高市早苗 首相は３日、 熊本地震 の被災地を視察した。首相は 熊本市 内で記者団の取材に応じ...,2026-08-04 05:00:00+09:00,https://www.asahi.com/articles/DA3S16519529.html,False,NaN
2,3,高市首相が熊本視察、予備費200億円超支出へ 「激甚災害」指定も,高市早苗 首相は3日、 熊本地震 の被災地を視察した。首相は 熊本市 内で記者団の取材に応じ...,2026-08-03 20:50:30+09:00,https://www.asahi.com/articles/ASV833TR3V83UTF...,False,NaN
3,4,災害時の死者名の公表、自治体が判断 熊本県は「遺族の同意」が前提,災害で犠牲になった人の氏名は公表されるのかどうか。最大震度7を観測した 熊本地震 で亡くなっ...,2026-08-03 18:15:00+09:00,https://www.asahi.com/articles/ASV832JLDV83UTI...,False,NaN
4,5,【31日の詳報】10年前は自宅全壊、今度は母を失う 救助「終了」,熊本県 を震源とする最大震度7の地震が28日午後4時半ごろ、発生しました。 イオンモール 熊...,2026-08-03 14:32:19+09:00,https://www.asahi.com/articles/ASV831RG5V83DIF...,False,NaN
...,...,...,...,...,...,...,...
1079,1080,新潟県内の住宅被害、新たに712棟 能登半島地震で,能登半島地震で、新潟県は24日、県内の住宅被害が前日より712棟増えて8720棟になったと発...,NaT,https://www.asahi.com/articles/ASS1S73G1S1SUOH...,True,404 Client Error: Not Found for url: https://w...
1080,1081,予備費４７億円支出決定 首相「プッシュ型支援を加速」 能登半島地震,政府は９日の閣議で、能登半島地震の被災地への物資支援のため、今年度予算の予備費から４７億４千...,NaT,https://www.asahi.com/articles/DA3S15834295.html,True,404 Client Error: Not Found for url: https://w...
1081,1082,北陸電力、能登半島地震で特別損失計上へ 社長「復旧復興めざす」,北陸電力の松田光司社長は1月31日、富山市の本社で記者会見し、能登半島地震でまだ残る奥能登地...,NaT,https://www.asahi.com/articles/ASS1066YXS10PIS...,True,404 Client Error: Not Found for url: https://w...
1082,1083,トラブル続発の志賀原発、報道陣に公開 能登半島地震から2カ月で初,能登半島地震の影響でトラブルが相次いだ北陸電力志賀原発（石川県）の内部が7日、地震後はじめて...,NaT,https://www.asahi.com/articles/ASS376GMKS37ULB...,True,404 Client Error: Not Found for url: https://w...


In [6]:
output_path = Path("asahi_能登半島地震.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"保存しました: {output_path.resolve()}")


保存しました: /Users/tj/IdeaProjects/sample/get-news/asahi_能登半島地震.csv
